# Cybertron - EN3150 Assignment 03


In [1]:
import os
if not os.path.exists('requirements.txt'):
    !git clone -q https://github.com/Isira-Himal-Perera/CNN-for-Edge-Image-Classification.git repo
    %cd repo
!pip install -q -r requirements.txt
print('working directory:', os.getcwd())

fatal: destination path 'repo' already exists and is not an empty directory.
/content/repo
working directory: /content/repo


In [2]:
from pathlib import Path
import torch
from IPython.display import Image, display

from src.data import get_data, make_loaders, SEED
from src.models import EdgeCNN, count_params, estimate_macs, load_pretrained, unfreeze_last_block
from src.experiment import seed_everything, optimizer_for, run_epochs, evaluate, describe_run

%matplotlib inline
import matplotlib.pyplot as plt

OUT = Path('results_notebook')
(OUT / 'models').mkdir(parents=True, exist_ok=True)
(OUT / 'plots').mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| torch:', torch.__version__)

device: cuda | torch: 2.11.0+cu130


In [3]:
splits, classes, metadata = get_data('digits')
loaders = make_loaders(splits, batch_size=64)
image_size = splits['train'][0].shape[-1]

for name, (x, y) in splits.items():
    print(f'{name:5s}: {len(y):4d} images ({100 * len(y) / 1797:.1f}%)')
print('classes:', classes)

x, y = splits['train']
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for ax, img, label in zip(axes.flat, x[:16], y[:16]):
    ax.imshow(img[0], cmap='gray')
    ax.set_title(str(int(label)))
    ax.axis('off')
fig.suptitle('training samples')
plt.show()

train: 1257 images (69.9%)
val  :  270 images (15.0%)
test :  270 images (15.0%)
classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']


In [4]:
model_a = EdgeCNN(len(classes), lightweight=False)
model_b = EdgeCNN(len(classes), lightweight=True)

for name, m in [('Model A (standard)', model_a), ('Model B (separable)', model_b)]:
    params = count_params(m)
    print(f'{name:22s} params={params:>7,d}  MACs/image={estimate_macs(m, image_size):,}')

assert count_params(model_b) <= 100_000, 'Model B exceeds the 100k parameter limit'
print('Model B is within the 100k parameter limit')

Model A (standard)     params= 26,218  MACs/image=2,804,032
Model B (separable)    params=  5,813  MACs/image=396,608
Model B is within the 100k parameter limit
